# Period 2: Login and logout

Practise bench for the **`/login` check**. The form, the cookie and `/logout` are in the app files, and the check cell at the end tests them.

**How this notebook works**
1. **Practise here first.** Write and test each piece in the cells below. A cell that says **TODO** is yours to write.
2. Click a cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
3. A check cell prints **✓** when you are right, or **✗** and a hint when you are not. Fix the cell above it and run both again.
4. When every check shows ✓, **copy your code into the real app files** (the last section tells you where).

If a cell shows a red error such as `no such table`, you skipped a TODO cell above it. Run that one first.

The database in this notebook is a **practice copy that lives in memory**. It is not your real `ais.db`, so you cannot break anything.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")   # a practice database that lives in memory
db.row_factory = sqlite3.Row       # so each row can be read like a dictionary: row["name"]

def show(sql, params=()):
    """Run a SELECT and print the rows as a small table. (Given: you do not need to read this.)"""
    cur = db.execute(sql, params)
    rows = cur.fetchall()
    names = [d[0] for d in cur.description]
    widths = [max([len(str(n))] + [len(str(r[n])) for r in rows]) for n in names]
    print("  ".join(str(n).ljust(w) for n, w in zip(names, widths)))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print("  ".join(str(r[n]).ljust(w) for n, w in zip(names, widths)))
    print("(" + str(len(rows)) + " rows)")

def check(ok, good, hint):
    """Print a tick or a cross. (Given.)"""
    if ok:
        print("✓ " + good)
    else:
        print("✗ " + hint)


# A practice users table with the same fake accounts as the app. (Given.)
db.executescript("""
CREATE TABLE users (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE NOT NULL, password TEXT NOT NULL, role TEXT NOT NULL);
INSERT INTO users (username, password, role) VALUES ('admin', 'practice-only-password', 'admin');
INSERT INTO users (username, password, role) VALUES ('aisha', 'sunflower22', 'user');
INSERT INTO users (username, password, role) VALUES ('ben', 'basketball7', 'user');
INSERT INTO users (username, password, role) VALUES ('chloe', 'mango-lassi', 'user');
""")
show("SELECT id, username, role FROM users")

## 1. Build the query from the inputs

When someone signs in, the route gets a **username** and a **password** from the form. It builds a query that looks for a row with both, by **joining pieces of text with `+`**.

For `aisha` and `sunflower22`, the finished query must read:

```
SELECT username, role FROM users WHERE username = 'aisha' AND password = 'sunflower22'
```

The single quotes around each value are part of the text you join.

### TODO 1: build `query`

In [ ]:
username = "aisha"
password = "sunflower22"

# TODO 1: build query by joining pieces of text with +
query = ""

print(query)

In [ ]:
expected = "SELECT username, role FROM users WHERE username = 'aisha' AND password = 'sunflower22'"
check(" ".join(query.split()) == expected, "Your query matches the expected text.",
      "Your query does not match yet. Print it and compare it with the expected text above: spaces and quotes both matter.")

## 2. Run the query: `check_login`

`fetchone()` gives the **first matching row**, or `None` if nothing matched. If the query is broken, `try / except` turns the error into "no match" instead of crashing.

### TODO 2: finish the function

Build the query from the function's inputs, then run it.

In [ ]:
def check_login(username, password):
    # TODO 2a: build the query, the same way as above, using username and password
    query = ""

    try:
        # TODO 2b: run the query and keep the first matching row
        row = None
    except sqlite3.Error:
        row = None
    return row

In [ ]:
r1 = check_login("aisha", "sunflower22")
check(r1 is not None and r1["username"] == "aisha", "aisha with the right password: a row comes back.",
      "aisha with the right password should give a row, but you got " + str(r1) + ".")
check(check_login("aisha", "wrong") is None, "aisha with a wrong password: None.",
      "A wrong password should give None.")
check(check_login("nobody", "x") is None, "An unknown name: None.",
      "An unknown username should give None.")
r2 = check_login("ben", "basketball7")
check(r2 is not None and r2["role"] == "user", "ben signs in and his role is user.",
      "ben with the right password should give a row with role user.")

Remember the line that builds the query. **In Period 4 you will test how safe it is.**

## 3. What a notebook cannot do

A notebook cannot show a page, so two parts of the login live only in the app:

- **The cookie.** When `check_login` finds a row, the route stores `uid = the username` in a **cookie**, a small note the browser keeps and sends back with every request. `current_user()` reads it.
- **Logout.** `/logout` deletes the cookie, so the browser forgets who you are.

## 4. Move it into the real app

| What | From this notebook | Into |
| --- | --- | --- |
| The sign-in form | (not in this notebook) | `templates/login.html`, in place of the TODO note |
| The query, `try / except`, the error message, the cookie | your TODO 2 code, plus the guide's cookie lines | `main.py`, in `/login` (replace the TODO and the old `return render_template("login.html", error="Sign-in is not built yet.")` line) |
| Logout | the guide's code | `main.py`, in `/logout` (replace the TODO and the old return line) |

Then run the check cell below. It tests your real files with a practice copy of the data.

In [ ]:
import os, shutil, tempfile, importlib

def load_app():
    """Load your main.py. (Given: you do not need to read this.)"""
    import main
    importlib.reload(main)
    return main

def app_with_copy_of_db(main):
    """Point the app at a COPY of your ais.db, so these checks never change your real data."""
    tmp = os.path.join(tempfile.mkdtemp(), "check.db")
    if os.path.exists("ais.db"):
        shutil.copy("ais.db", tmp)
    main.DB_PATH = tmp
    main.init_db()          # makes the users table if it is missing
    return main

In [ ]:
def check_my_login():
    """Check your real login and logout. (Given: you do not need to read this.)"""
    main = app_with_copy_of_db(load_app())
    c = main.app.test_client()

    page = c.get("/login").get_data(as_text=True)
    check('<form' in page and 'name="username"' in page and 'name="password"' in page,
          "The sign-in page has a form with a username and a password box.",
          "login.html needs a <form> with two inputs named username and password.")

    r = c.post("/login", data={"username": "aisha", "password": "sunflower22"})
    cookies = " ".join(r.headers.getlist("Set-Cookie"))
    check(r.status_code == 302 and "uid=aisha" in cookies,
          "aisha signs in: your app sets the cookie uid=aisha and redirects.",
          "A right password should set the cookie uid=aisha and redirect. Got status " + str(r.status_code) + ". Check /login in main.py: is your code above the old return line?")

    r = c.post("/login", data={"username": "aisha", "password": "wrong"})
    check(r.status_code == 200 and "Incorrect username or password" in r.get_data(as_text=True),
          "A wrong password shows Incorrect username or password.",
          "A wrong password should show the page again with the error message.")

    r = c.get("/logout")
    cookies = " ".join(r.headers.getlist("Set-Cookie"))
    check("uid=;" in cookies and r.status_code == 302,
          "/logout deletes the uid cookie and redirects to the sign-in page.",
          "/logout should delete the uid cookie. Is your code above the old return line?")

check_my_login()